Conseiller SAV
      │
      │ tape CMD-1042
      ▼
"CMD-1042"
      │
      ▼
Service
order_id = "CMD-1042"
      │
      ▼
Repository
order_id = "CMD-1042"
      │
      ▼
Session
order_id = "CMD-1042"
      │
      ▼
PostgreSQL

## Pipeline d'une demande SAV : comment `CMD-1042` traverse l'application

Imaginons qu'un conseiller SAV cherche une commande.

### 1. Le conseiller saisit le numéro de commande

Le conseiller tape :

`CMD-1042`

À ce moment-là, ce n'est encore qu'une simple chaîne de caractères, un `str`.

On peut l'imaginer comme :

`saisie_sav = "CMD-1042"`

La donnée existe donc simplement dans une variable Python.

---

### 2. La donnée est envoyée au Service

L'application appelle ensuite une méthode du Service :

`service.get_order("CMD-1042")`

Le Service possède une méthode qui ressemble à :

`get_order(self, order_id: str)`

Python prend donc la valeur `"CMD-1042"` et la place dans le paramètre `order_id`.

À cet instant, dans le Service :

`order_id = "CMD-1042"`

Le Service ne va pas lui-même chercher dans PostgreSQL.

Son rôle est plutôt de gérer la logique métier et de transmettre la demande à la bonne couche.

Il dit en quelque sorte :

> "On me demande la commande CMD-1042. Je vais demander au Repository de la récupérer."

---

### 3. Le Service transmet l'identifiant au Repository

Le Service appelle par exemple :

`repository.get_by_id(order_id)`

Comme `order_id` contient `"CMD-1042"`, cela revient à faire :

`repository.get_by_id("CMD-1042")`

Le Repository possède lui aussi un paramètre :

`get_by_id(self, order_id: str)`

Donc la même valeur arrive maintenant dans le Repository :

`order_id = "CMD-1042"`

Le Repository est la couche spécialisée dans l'accès aux données.

Il sait comment interroger la base, mais il ne se connecte pas directement à PostgreSQL : il utilise une Session SQLAlchemy.

---

### 4. Le Repository transmet l'identifiant à la Session

Le Repository fait quelque chose comme :

`self.session.execute(...)`

Dans la requête SQLAlchemy, il utilise la valeur reçue :

`OrderRow.order_id == order_id`

Comme `order_id` vaut `"CMD-1042"`, la requête signifie conceptuellement :

`chercher la ligne où order_id = "CMD-1042"`

La Session reçoit donc la requête préparée par le Repository.

La Session représente le canal de communication utilisé par SQLAlchemy pour parler à PostgreSQL.

---

### 5. PostgreSQL cherche la commande

PostgreSQL reçoit finalement une requête qui revient conceptuellement à :

`SELECT ... FROM orders WHERE order_id = 'CMD-1042'`

PostgreSQL cherche alors dans la table `orders`.

Il peut trouver par exemple :

- `order_id = "CMD-1042"`
- `customer_id = "CUS-001"`
- `status = "SHIPPED"`
- `expected_delivery = ...`

Le conseiller SAV avait uniquement fourni `"CMD-1042"`.

Les autres informations viennent de la base de données.

---

## Vue complète de la descente de la donnée

`Conseiller SAV`

↓ tape

`"CMD-1042"`

↓

`input()` / variable Python

`saisie_sav = "CMD-1042"`

↓

appel du Service

`service.get_order(saisie_sav)`

↓

dans le Service

`order_id = "CMD-1042"`

↓

appel du Repository

`repository.get_by_id(order_id)`

↓

dans le Repository

`order_id = "CMD-1042"`

↓

création de la requête SQLAlchemy

`OrderRow.order_id == order_id`

↓

Session SQLAlchemy

↓

PostgreSQL

↓

recherche :

`order_id = "CMD-1042"`

---

## Puis la donnée fait le trajet inverse

PostgreSQL trouve les données de la commande.

↓

Le résultat revient dans la Session.

↓

Le Repository récupère les données SQLAlchemy.

↓

Le Repository construit un objet métier `Order`.

Par exemple :

`Order(order_id="CMD-1042", customer_id="CUS-001", status="SHIPPED", ...)`

↓

Le Repository retourne cet objet au Service.

↓

Le Service retourne l'objet à la couche qui l'a appelé.

↓

Le conseiller SAV peut finalement voir les informations de la commande.

---

## La chose essentielle à retenir

La valeur `"CMD-1042"` ne se déplace pas toute seule.

À chaque étape, une fonction reçoit la valeur dans un paramètre puis la transmet à une autre fonction.

Le mécanisme est toujours le même :

`fonction(valeur)`

↓

`def fonction(parametre):`

↓

`parametre = valeur`

Dans notre exemple :

`service.get_order("CMD-1042")`

donne dans le Service :

`order_id = "CMD-1042"`

Puis :

`repository.get_by_id(order_id)`

donne dans le Repository :

`order_id = "CMD-1042"`

La même donnée traverse donc progressivement les différentes couches de l'application.

In [ ]:
from pydantic import BaseModel


# ============================================================
# 1. MODELE METIER
# ============================================================

class Order(BaseModel):
    """
    Modèle de commande utilisé dans la simulation.
    """

    order_id: str
    customer_id: str
    status: str


# ============================================================
# 2. SESSION
# ============================================================

class Session:
    """
    Simule une session de lecture à partir d'un dictionnaire Python.

    Dans l'application, les repositories utilisent des sessions SQLAlchemy
    connectées à PostgreSQL.
    """

    def __init__(self) -> None:

        # Données en mémoire représentant la table orders.
        self.database: dict[str, dict[str, str]] = {
            "CMD-1042": {
                "order_id": "CMD-1042",
                "customer_id": "CUS-001",
                "status": "SHIPPED",
            },
            "CMD-1050": {
                "order_id": "CMD-1050",
                "customer_id": "CUS-004",
                "status": "DELIVERED",
            },
        }

    def execute(
        self,
        order_id: str,
    ) -> dict[str, str] | None:

        print("\n--- SESSION ---")
        print("La Session reçoit :", order_id)

        # Simulation de :
        #
        # SELECT *
        # FROM orders
        # WHERE order_id = ...
        row = self.database.get(order_id)

        print("La base renvoie :", row)

        return row


# ============================================================
# 3. REPOSITORY
# ============================================================

class OrderRepository:
    """
    Responsable de l'accès aux données des commandes.
    """

    def __init__(
        self,
        session: Session,
    ) -> None:

        # Le repository conserve la session reçue en paramètre.
        self.session = session

        print("\nRepository créé")
        print("Il reçoit une Session :", self.session)

    def get_by_id(
        self,
        order_id: str,
    ) -> Order | None:

        print("\n--- REPOSITORY ---")
        print("Le Repository reçoit :", order_id)

        # Le Repository demande les données à la Session.
        row = self.session.execute(order_id)

        # L'absence de commande est représentée par None.
        if row is None:
            print("Le Repository n'a trouvé aucune commande.")
            return None

        # Conversion du dictionnaire renvoyé par la session en modèle Order.
        order = Order(
            order_id=row["order_id"],
            customer_id=row["customer_id"],
            status=row["status"],
        )

        print("Le Repository construit le modèle Order :")
        print(order)

        return order


# ============================================================
# 4. UNIT OF WORK
# ============================================================

class UnitOfWork:
    """
    Prépare la Session et les Repositories.
    """

    def __init__(self) -> None:

        print("\n--- UNIT OF WORK ---")
        print("Création du UnitOfWork")

        # Création d'une Session.
        self.session = Session()

        print("Session créée :", self.session)

        # Le repository utilise la session de cette unité de travail.
        self.orders = OrderRepository(
            self.session
        )

        print("Repository disponible dans : self.orders")


# ============================================================
# 5. SERVICE METIER
# ============================================================

class OrderOpsService:
    """
    Contient la logique métier.
    """

    def __init__(
        self,
        uow_factory: type[UnitOfWork],
    ) -> None:

        # La classe UnitOfWork sert de fabrique ; aucune instance n'est encore créée.
        self.uow_factory = uow_factory

        print("\n--- CREATION SERVICE ---")
        print("Le Service reçoit :", self.uow_factory)

    def get_order(
        self,
        order_id: str,
    ) -> Order:

        print("\n--- SERVICE ---")
        print("Le Service reçoit :", order_id)

        # Suppression des espaces autour de l'identifiant de commande.
        order_id = order_id.strip()

        print("Après nettoyage :", order_id)

        # L'appel de la fabrique crée une unité de travail pour cette requête.
        uow = self.uow_factory()

        # Recherche par identifiant via le repository de l'unité de travail.
        order = uow.orders.get_by_id(
            order_id
        )

        # Une commande absente déclenche une erreur dans le service.
        if order is None:
            raise ValueError(
                f"Commande {order_id} introuvable"
            )

        print("\nLe Service reçoit l'objet Order :")
        print(order)

        return order


# ============================================================
# 6. CONSTRUCTION DE L'APPLICATION
# ============================================================

# Le service reçoit la classe UnitOfWork comme fabrique.
service = OrderOpsService(
    UnitOfWork
)


# ============================================================
# 7. SIMULATION DE L'ADV / SAV
# ============================================================

print("\n==============================")
print("INTERFACE ADV / SAV")
print("==============================")

# La saisie du numéro de commande bloque l'exécution jusqu'à validation.
saisie_adv: str = input(
    "Numéro de commande : "
)


print("\nL'ADV a saisi :", saisie_adv)


# ============================================================
# 8. APPEL DU SERVICE
# ============================================================

commande: Order = service.get_order(
    saisie_adv
)


# ============================================================
# 9. RESULTAT POUR L'ADV
# ============================================================

print("\n==============================")
print("RESULTAT AFFICHÉ À L'ADV")
print("==============================")

print("Commande :", commande.order_id)
print("Client :", commande.customer_id)
print("Statut :", commande.status)